# 01 · Quickstart — a first DFXM image in five lines

Dark-field X-ray microscopy (DFXM) maps lattice distortion deep inside crystals by imaging a single Bragg reflection through an objective lens. `dfxm_geo` simulates it in two stages: a reciprocal-space **resolution kernel** (which local q-deviations the instrument accepts) and a direct-space **geometrical-optics pass** (what each detector pixel collects through that acceptance). Physics: [Borgi 2024](../docs/references.md#borgi-2024) §2; all citations resolve on the [references page](../docs/references.md).

![DFXM optics schematic](../docs/img/fig_3_1_microscope_schematic.png)

Stage 1 runs once per reflection (`dfxm-bootstrap`: Monte-Carlo ray sampling → an `.npz` lookup table); stage 2 runs per frame (`dfxm-forward` / `run_simulation`). The setup cell builds the tutorial kernel at production sampling (10⁸ rays, about a minute on first run; cached in `examples/kernel/` afterwards) and points the in-process lookup there.

In [ ]:
%matplotlib inline
import subprocess
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

import dfxm_geo
import dfxm_geo.direct_space.forward_model as fm

HERE = Path.cwd()
assert HERE.name == "examples", "Run this notebook from the examples/ folder"
IMG, OUT, KERNEL_DIR = HERE / "img", HERE / "out_01", HERE / "kernel"
for d in (IMG, OUT, KERNEL_DIR):
    d.mkdir(exist_ok=True)
fm.pkl_fpath = KERNEL_DIR.resolve().as_posix() + "/"  # kernel lookup -> examples/kernel/

KERNEL_FILE = KERNEL_DIR / "Resq_i_h-1_k1_l-1_17keV_tutorial.npz"
if not KERNEL_FILE.exists():
    boot = OUT / "bootstrap.toml"
    boot.write_text(
        "[reciprocal]\nNrays = 100_000_000\nbatch_size = 20_000_000\nseed = 0\nbeamstop = false\n",
        encoding="utf-8",
    )
    subprocess.run(
        [
            sys.executable,
            "-c",
            "from dfxm_geo.reciprocal_space.kernel import cli_main; raise SystemExit(cli_main())",
            "--config",
            str(boot),
            "--output",
            str(KERNEL_FILE),
        ],
        check=True,
    )
print("dfxm_geo", dfxm_geo.__version__, "| kernel:", KERNEL_FILE.name)

## An empty TOML is a valid experiment

Since v2.0.0 every config key has a physically sensible default: one edge dislocation centred in single-crystal aluminium, imaged on the (−1, 1, −1) reflection at 17 keV. An empty file runs out of the box; any key you write overrides exactly one default.

In [ ]:
from dfxm_geo.pipeline import SimulationConfig, run_simulation

(OUT / "quickstart.toml").write_text("", encoding="utf-8")
cfg = SimulationConfig.from_toml(OUT / "quickstart.toml")
result = run_simulation(cfg, OUT)
print("wrote", result["h5_path"])

## What got written

`run_simulation` produced a BLISS-style master file with one NXentry per scan — `/1.1` is the dislocated crystal, `/2.1` the perfect-crystal reference (written when `[io] include_perfect_crystal = true`, the default) — with detector frames in per-scan files reached through HDF5 external links. Layout details: [docs/output-format.md](../docs/output-format.md).

In [ ]:
with h5py.File(result["h5_path"], "r") as f:
    disloc = f["/1.1/instrument/dfxm_sim_detector/data"][0]
    perfect = f["/2.1/instrument/dfxm_sim_detector/data"][0]

vmax = max(disloc.max(), perfect.max())
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, im, title in [
    (axes[0], disloc, "centred edge dislocation"),
    (axes[1], perfect, "perfect crystal"),
]:
    ax.imshow(im, cmap="magma", norm=PowerNorm(0.45, vmin=0.0, vmax=vmax))
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(IMG / "01_quickstart_preview.png", dpi=110, bbox_inches="tight")

## Next

- [02 · Reciprocal space](02_reciprocal_space.ipynb) — what the resolution kernel is.
- [03 · Dislocations & contrast](03_dislocations_and_contrast.ipynb) — character, weak beam, COM ≈ −qi.
- [04 · Oblique & reflections](04_oblique_and_reflections.ipynb) — mounts, reflection tables, g·b invisibility.
- [05 · Identification at scale](05_identification_at_scale.ipynb) — labelled ML datasets and throughput.